# 🎓 PathOptLearn — Full System Integration

This notebook combines **all components** into a single API server:
- Question Generation (Mistral-7B LoRA)
- Answer Evaluation (Mistral-7B LoRA)
- Video Recommender (Gemini + YouTube)
- Progress Tracking (SQLite)

Run all cells in order, then use the updated Streamlit app (Cell 10) to connect.

In [ ]:
# ── CELL 1: Install Dependencies ───────────────────────────
!pip install -q transformers accelerate peft bitsandbytes torch
!pip install -q fastapi uvicorn pyngrok nest-asyncio
!pip install -q youtube-search-python yt-dlp google-generativeai

In [ ]:
# ── CELL 2: Mount Drive & Load Question Generation Model ──
import torch, json, re, os, sqlite3, tempfile, urllib.request
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datetime import datetime
from collections import Counter

from google.colab import drive
drive.mount('/content/drive')

QG_MODEL_PATH = "/content/drive/MyDrive/question-generation-mistral"
BASE_MODEL    = "mistralai/Mistral-7B-v0.1"

print("Loading question generation model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, QG_MODEL_PATH)
model.eval()
print("✅ Question generation model ready!")

In [ ]:
# ── CELL 3: Load Evaluation Model ─────────────────────────
EVAL_MODEL_PATH = "/content/drive/MyDrive/adaptive-learning-evaluation-FINAL"

print("Loading evaluation model...")
eval_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
eval_tokenizer.pad_token = eval_tokenizer.eos_token
eval_tokenizer.padding_side = "right"

eval_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True,
)
eval_model = PeftModel.from_pretrained(eval_base_model, EVAL_MODEL_PATH)
eval_model.eval()
print("✅ Evaluation model ready!")

In [ ]:
# ── CELL 4: Configure Gemini for Recommender ─────────────
import google.generativeai as genai

# ⚠️ Replace with your Gemini API key (https://aistudio.google.com/apikey)
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-1.5-flash")
print("✅ Gemini configured!")

In [ ]:
# ── CELL 5: Question Generation & Evaluation Functions ────
# (Same as APIs.ipynb — parse, generate, evaluate)

def parse_generated_question(generated_text):
    try:
        if "### Question:" not in generated_text: return None
        parts = generated_text.rsplit("### Question:", 1)[1]
        if "### Choices:" in parts:
            question_text = parts.split("### Choices:")[0].strip()
            rest = parts.split("### Choices:")[1]
        else:
            lines = [l.strip() for l in parts.strip().split('\n') if l.strip()]
            if not lines: return None
            question_text, rest = lines[0], '\n'.join(lines[1:])
        if "### Correct Answer:" in rest:
            choices_block = rest.split("### Correct Answer:")[0]
            correct_block = rest.split("### Correct Answer:")[1]
        else:
            choices_block, correct_block = rest, ""
        choices = []
        for line in choices_block.split('\n'):
            m = re.match(r'^[A-Da-d1-4][)\\.\\:]\\s*(.+)', line.strip())
            if m: choices.append(m.group(1).strip())
        if len(choices) < 2: return None
        correct_idx = 0
        if correct_block.strip():
            letter = correct_block.strip()[0].upper()
            if letter in "ABCD": correct_idx = ord(letter) - ord('A')
            elif letter in "1234": correct_idx = int(letter) - 1
        correct_idx = min(correct_idx, len(choices) - 1)
        if not question_text: return None
        return {"question": question_text, "choices": choices, "correct": correct_idx}
    except: return None

def generate_questions(transcript, num_questions=5, temperature=0.8):
    chunk_size, overlap = 2800, 200
    chunks, start = [], 0
    while start < len(transcript):
        chunks.append(transcript[start:start+chunk_size])
        start += chunk_size - overlap
    qpc = [num_questions // len(chunks)] * len(chunks)
    for i in range(num_questions % len(chunks)): qpc[i] += 1
    all_questions, q_counter = [], 0
    for chunk_idx, (chunk, n_q) in enumerate(zip(chunks, qpc)):
        for i in range(n_q):
            q_counter += 1
            parsed = None
            for attempt in range(2):
                prompt = f"### Passage:\n{chunk.strip()}\n\n### Task:\nGenerate question #{q_counter} - a multiple-choice question.\n\n### Question:"
                inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
                with torch.no_grad():
                    outputs = model.generate(**inputs, max_new_tokens=250, temperature=temperature+(attempt*0.1), do_sample=True, top_p=0.95, pad_token_id=tokenizer.eos_token_id)
                parsed = parse_generated_question(tokenizer.decode(outputs[0], skip_special_tokens=True))
                if parsed: break
            if parsed: all_questions.append(parsed)
    return all_questions

def run_evaluation(transcript, questions, student_answers, preference):
    context = transcript.strip()[:2800]
    qa = ''
NGROK_AUTH_TOKEN = "3AIhGm2Ih8LkOrw7B44HG8NKU5S_7C516SgaGBYDDWxdALaaB"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8002)

print("\n" + "=" * 60)
print(f"  ✅ Progress API is live at: {public_url}")
print(f"  Health check:     {public_url}/health")
print(f"  Save session:     POST {public_url}/progress/save")
print(f"  Get history:      GET  {public_url}/progress/{{student_id}}")
print(f"  Get watched:      GET  {public_url}/progress/{{student_id}}/watched")
print(f"  Get dashboard:    GET  {public_url}/progress/{{student_id}}/dashboard")
print("=" * 60)


def run_server():
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8002)


print("\nStarting server in background thread...")
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Server started! This cell will block to keep it alive.")
print("Interrupt this cell to stop the server.\n")

try:
    while True:
        time.sleep(3600)
except KeyboardInterrupt:
    print("\nServer stopped.")
    for i, (q, a) in enumerate(zip(questions, student_answers), 1):
        qa += f"Q{i}: {q['question']}\nStudent Answer: {a}\n\n"
    prompt = f"### Domain: [GENERAL]\n### Context:\n{context}\n\n### Assessment:\n{qa}### User Preference: {preference}\n\n### Evaluation:"
    inputs = eval_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        outputs = eval_model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True, top_p=0.95, pad_token_id=eval_tokenizer.eos_token_id)
    text = eval_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Evaluation:" in text:
        after = text.split("### Evaluation:")[1].strip()
        if "### Recommendation:" in after:
            return {"evaluation": after.split("### Recommendation:")[0].strip(), "recommendation": after.split("### Recommendation:")[1].strip()}
        return {"evaluation": after, "recommendation": ""}
    return {"evaluation": text[-300:], "recommendation": ""}

print("✅ Question generation & evaluation functions ready!")

In [ ]:
# ── CELL 6: Recommender Functions ─────────────────────────
from youtubesearchpython import VideosSearch
import yt_dlp

def search_youtube(topic, max_results=10):
    results = VideosSearch(topic, limit=max_results).result()["result"]
    return [{
        "id": r.get("id",""), "title": r.get("title",""), "duration": r.get("duration",""),
        "views": r.get("viewCount",{}).get("short","N/A"), "channel": r.get("channel",{}).get("name","Unknown"),
        "description": (r.get("descriptionSnippet",[{}])[0].get("text","") if r.get("descriptionSnippet") else ""),
        "url": f"https://www.youtube.com/watch?v={r.get('id','')}",
        "has_captions": False, "transcript_snippet": "",
    } for r in results]

def get_video_captions(video_id):
    try:
        with yt_dlp.YoutubeDL({'skip_download':True,'writesubtitles':True,'writeautomaticsub':True,'subtitleslangs':['en'],'quiet':True,'no_warnings':True}) as ydl:
            info = ydl.extract_info(f"https://www.youtube.com/watch?v={video_id}", download=False)
            sub_url = None
            if 'subtitles' in info and 'en' in info.get('subtitles',{}): sub_url = info['subtitles']['en'][0]['url']
            elif 'automatic_captions' in info and 'en' in info.get('automatic_captions',{}): sub_url = info['automatic_captions']['en'][0]['url']
            if not sub_url: return ""
            with urllib.request.urlopen(sub_url) as resp:
                data = json.loads(resp.read())
            parts = [' '.join(seg.get('utf8','') for seg in e['segs']).strip() for e in data.get('events',[]) if 'segs' in e]
            return ' '.join(p for p in parts if p)[:2000]
    except: return ""

def recommend_next_video(ctx):
    topic, passed, weak = ctx.get('topic',''), ctx.get('passed',False), ctx.get('weak_areas',[])
    watched = set(ctx.get('watched_video_ids',[]))
    query = f"{topic} advanced tutorial" if passed else f"{topic} {' '.join(weak[:2])} {ctx.get('student_level','intermediate')} explained"
    videos = [v for v in search_youtube(query, 10) if v['id'] not in watched][:5]
    if not videos: return {"recommended_video": None, "alternatives": [], "message": "No new videos found."}
    for v in videos:
        t = get_video_captions(v['id'])
        if t: v['has_captions'], v['transcript_snippet'] = True, t
    # LLM ranking
    descs = '\n\n'.join([f"Video {i+1}:\n  Title: {v['title']}\n  Channel: {v['channel']}\n  Duration: {v['duration']}\n  Content: {v['transcript_snippet'][:300] if v['has_captions'] else v['description'][:200]}" for i,v in enumerate(videos)])
    prompt = f"""Pick the BEST next video for a student.\nStudent: level={ctx.get('student_level','intermediate')}, passed={passed}, weak_areas={weak}, topic={topic}\n\nCandidates:\n{descs}\n\nRespond ONLY with JSON: {{"best": <number>, "reason": "...", "alts": [<number>]}}"""
    try:
        resp = json.loads(re.sub(r'^```json\s*|\s*```$', '', gemini_model.generate_content(prompt).text.strip()))
        best = videos[resp['best']-1]
        alts = [{'id':videos[a-1]['id'],'title':videos[a-1]['title'],'url':videos[a-1]['url'],'reason':''} for a in resp.get('alts',[]) if 0<a<=len(videos)]
        return {"recommended_video": {"id":best['id'],"title":best['title'],"url":best['url'],"reason":resp['reason']}, "alternatives": alts}
    except:
        return {"recommended_video": {"id":videos[0]['id'],"title":videos[0]['title'],"url":videos[0]['url'],"reason":"Fallback"}, "alternatives": []}

print("✅ Recommender functions ready!")

In [ ]:
# ── CELL 7: Progress Tracking (SQLite) ────────────────────
DB_PATH = "pathoptlearn_progress.db"

def get_db():
    conn = sqlite3.connect(DB_PATH); conn.row_factory = sqlite3.Row; return conn

def init_db():
    conn = get_db()
    conn.execute("""CREATE TABLE IF NOT EXISTS student_progress (
        id INTEGER PRIMARY KEY AUTOINCREMENT, student_id TEXT NOT NULL,
        video_id TEXT NOT NULL, video_title TEXT NOT NULL,
        score_raw INTEGER NOT NULL, score_total INTEGER NOT NULL, score_pct REAL NOT NULL,
        passed BOOLEAN NOT NULL, evaluation_text TEXT DEFAULT '', recommendation TEXT DEFAULT '',
        timestamp TEXT NOT NULL, topic TEXT DEFAULT '', weak_areas TEXT DEFAULT '[]')""")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_student ON student_progress(student_id)")
    conn.commit(); conn.close()

init_db()

def save_session(student_id, video_id, video_title, score_raw, score_total, passed, evaluation_text='', recommendation='', topic='', weak_areas=None):
    pct = (score_raw/score_total*100) if score_total>0 else 0
    ts = datetime.utcnow().isoformat()+'Z'
    conn = get_db()
    cur = conn.execute("INSERT INTO student_progress (student_id,video_id,video_title,score_raw,score_total,score_pct,passed,evaluation_text,recommendation,timestamp,topic,weak_areas) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
        (student_id,video_id,video_title,score_raw,score_total,pct,passed,evaluation_text,recommendation,ts,topic,json.dumps(weak_areas or [])))
    sid = cur.lastrowid; conn.commit(); conn.close(); return sid

def get_student_history(student_id):
    conn = get_db()
    rows = conn.execute("SELECT * FROM student_progress WHERE student_id=? ORDER BY timestamp ASC",(student_id,)).fetchall()
    conn.close()
    return [dict(r) | {'passed': bool(r['passed']), 'weak_areas': json.loads(r['weak_areas']) if r['weak_areas'] else []} for r in rows]

def get_watched_videos(student_id):
    conn = get_db()
    ids = [r['video_id'] for r in conn.execute("SELECT DISTINCT video_id FROM student_progress WHERE student_id=?",(student_id,)).fetchall()]
    conn.close(); return ids

def get_dashboard_stats(student_id):
    h = get_student_history(student_id)
    if not h: return {"total_sessions":0,"total_videos":0,"avg_score_pct":0,"pass_rate":0,"score_history":[],"learning_path":[]}
    t = len(h); p = sum(1 for x in h if x['passed'])
    return {"total_sessions":t, "total_videos":len(set(x['video_id'] for x in h)),
        "avg_score_pct":round(sum(x['score_pct'] for x in h)/t,1), "pass_rate":round(p/t*100,1),
        "score_history":[{"score_pct":x['score_pct'],"timestamp":x['timestamp']} for x in h],
        "learning_path":[{"video_id":x['video_id'],"video_title":x['video_title'],"score_pct":x['score_pct'],"passed":x['passed']} for x in h]}

print("✅ Progress tracking ready!")

In [ ]:
# ── CELL 8: Combined FastAPI Server ──────────────────────
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import uvicorn, nest_asyncio, threading, time
from pyngrok import ngrok

nest_asyncio.apply()
app = FastAPI(title="PathOptLearn — Full API")

# --- Request/Response Models ---
class GenerateRequest(BaseModel):
    transcript: str; num_questions: Optional[int]=5; temperature: Optional[float]=0.8

class EvaluateRequest(BaseModel):
    transcript: str; questions: list; student_answers: list; preference: Optional[str]="videos"

class RecommendRequest(BaseModel):
    topic: str; student_level: Optional[str]="intermediate"; passed: Optional[bool]=False
    weak_areas: Optional[List[str]]=[]; preference: Optional[str]="videos"; watched_video_ids: Optional[List[str]]=[]

class SaveSessionRequest(BaseModel):
    student_id: str; video_id: str; video_title: str; score_raw: int; score_total: int
    passed: bool; evaluation_text: Optional[str]=""; recommendation: Optional[str]=""
    topic: Optional[str]=""; weak_areas: Optional[List[str]]=[]

# --- Endpoints ---
@app.get("/health")
def health(): return {"status": "ok", "services": ["generate","evaluate","recommend","progress"]}

@app.post("/generate")
def generate(req: GenerateRequest):
    if not req.transcript.strip(): raise HTTPException(400, "Transcript is empty.")
    questions = generate_questions(req.transcript, req.num_questions, req.temperature)
    return {"questions": questions, "count": len(questions)}

@app.post("/evaluate")
def evaluate(req: EvaluateRequest):
    if not req.transcript.strip(): raise HTTPException(400, "Transcript is empty.")
    if len(req.questions) != len(req.student_answers): raise HTTPException(400, "Length mismatch.")
    score = sum(1 for q,a in zip(req.questions,req.student_answers) if a==q['choices'][q['correct']])
    total = len(req.questions); passed = score >= total*0.6
    result = run_evaluation(req.transcript, req.questions, req.student_answers, req.preference)
    return {"evaluation":result['evaluation'],"recommendation":result['recommendation'],"score":score,"total":total,"passed":passed}

@app.post("/recommend")
def recommend(req: RecommendRequest):
    if not req.topic.strip(): raise HTTPException(400, "Topic is empty.")
    return recommend_next_video(req.dict())

@app.post("/progress/save")
def save_progress(req: SaveSessionRequest):
    sid = save_session(**req.dict())
    return {"session_id": sid, "message": "Saved."}

@app.get("/progress/{student_id}")
def get_progress(student_id: str): return {"student_id": student_id, "history": get_student_history(student_id)}

@app.get("/progress/{student_id}/watched")
def get_watched(student_id: str): return {"student_id": student_id, "watched_video_ids": get_watched_videos(student_id)}

@app.get("/progress/{student_id}/dashboard")
def get_dashboard(student_id: str): return {"student_id": student_id, **get_dashboard_stats(student_id)}

print("✅ Combined FastAPI app ready!")

In [ ]:
# ── CELL 9: Start Server + Ngrok ──────────────────────────
# ⚠️ Replace with your ngrok auth token
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)

print("\n" + "="*60)
print(f"  ✅ Full API is live at: {public_url}")
print(f"  Endpoints:")
print(f"    GET  /health")
print(f"    POST /generate")
print(f"    POST /evaluate")
print(f"    POST /recommend")
print(f"    POST /progress/save")
print(f"    GET  /progress/{{student_id}}")
print(f"    GET  /progress/{{student_id}}/watched")
print(f"    GET  /progress/{{student_id}}/dashboard")
print("="*60)
print("\nCopy the URL above into your Streamlit app's API_URL variable.\n")

def run_server():
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("Server started. Interrupt this cell to stop.\n")
try:
    while True: time.sleep(3600)
except KeyboardInterrupt:
    print("\nServer stopped.")

# 🖥️ Updated Streamlit App

Copy the code in the next cell into `youtube_transcript_app_3.py` to get the full integrated experience with:
- **Step 4**: Video recommendation after quiz evaluation
- **Sidebar**: Student progress dashboard
- **Session persistence**: Student ID tracking

In [ ]:
# ── CELL 10: Updated Streamlit App Code ───────────────────
# Save this as 'youtube_transcript_app_integrated.py' and run:
#   streamlit run youtube_transcript_app_integrated.py

STREAMLIT_CODE = '''
import streamlit as st
import re, json, os, tempfile, requests, yt_dlp, urllib.request

st.set_page_config(page_title="Adaptive Learning", page_icon="🎓", layout="wide")

# ── CONFIG ─────────────────────────────────────────────────
API_URL = "https://YOUR-NGROK-URL-HERE.ngrok-free.dev"  # ← paste from Cell 9

CACHE_DIR = os.path.join(tempfile.gettempdir(), "transcript_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# ── SIDEBAR: Student ID & Progress ────────────────────────
with st.sidebar:
    st.header("👤 Student Profile")
    student_id = st.text_input("Student ID:", value="student_1", key="student_id")
    st.session_state["student_id"] = student_id

    if st.button("📊 Load Progress"):
        try:
            resp = requests.get(f"{API_URL}/progress/{student_id}/dashboard", timeout=30)
            if resp.ok:
                dash = resp.json()
                st.metric("Videos Watched", dash.get("total_videos", 0))
                st.metric("Avg Score", f"{dash.get(\\"avg_score_pct\\", 0)}%")
                st.metric("Pass Rate", f"{dash.get(\\"pass_rate\\", 0)}%")
                if dash.get("learning_path"):
                    st.subheader("📚 Learning Path")
                    for step in dash["learning_path"]:
                        icon = "✅" if step["passed"] else "❌"
                        st.write(f"{icon} {step[\\"video_title\\"]} — {step[\\"score_pct\\"]:.0f}%")
        except Exception as e:
            st.error(f"Could not load progress: {e}")

    # Show watched videos
    try:
        resp = requests.get(f"{API_URL}/progress/{student_id}/watched", timeout=10)
        if resp.ok:
            watched = resp.json().get("watched_video_ids", [])
            st.session_state["watched_video_ids"] = watched
    except:
        st.session_state.setdefault("watched_video_ids", [])


# ── TRANSCRIPT HELPERS ──────────────────────────────────────
def extract_video_id(url):
    for pat in [r"(?:youtube\\.com\\/watch\\?v=)([^&]+)", r"(?:youtu\\.be\\/)([^?]+)", r"(?:youtube\\.com\\/embed\\/)([^?]+)"]:
        m = re.search(pat, url)
        if m: return m.group(1)
    return url.strip()

def get_transcript_captions(video_id):
    url = f"https://www.youtube.com/watch?v={video_id}"
    try:
        with yt_dlp.YoutubeDL({"skip_download":True,"writesubtitles":True,"writeautomaticsub":True,"subtitleslangs":["en"],"quiet":True,"no_warnings":True}) as ydl:
            info = ydl.extract_info(url, download=False)
            sub_url = None
            if "subtitles" in info and "en" in info["subtitles"]: sub_url = info["subtitles"]["en"][0]["url"]
            elif "automatic_captions" in info and "en" in info["automatic_captions"]: sub_url = info["automatic_captions"]["en"][0]["url"]
            if not sub_url: return None
            with urllib.request.urlopen(sub_url) as response:
                data = json.loads(response.read())
            return [{"text": " ".join(seg.get("utf8","") for seg in e["segs"]).strip(), "start": e.get("tStartMs",0)/1000} for e in data.get("events",[]) if "segs" in e and " ".join(seg.get("utf8","") for seg in e["segs"]).strip()]
    except: return None


# ── API HELPERS ─────────────────────────────────────────────
def call_api(endpoint, payload):
    try:
        resp = requests.post(f"{API_URL}/{endpoint}", json=payload, timeout=300)
        resp.raise_for_status()
        return resp.json(), None
    except Exception as e:
        return None, str(e)


# ── MAIN UI ─────────────────────────────────────────────────
st.title("🎓 Adaptive Learning System")
st.markdown("Watch → Quiz → Evaluate → Get Next Video Recommendation")
st.markdown("---")

# Step 1: Transcript
st.header("Step 1 — Extract Transcript")
video_input = st.text_input("YouTube URL or Video ID:", placeholder="WUvTyaaNkzM")
if st.button("🔍 Get Transcript", type="primary") and video_input:
    video_id = extract_video_id(video_input)
    with st.spinner("Processing..."):
        transcript = get_transcript_captions(video_id)
    if transcript:
        full_text = " ".join(t["text"] for t in transcript)
        st.session_state["transcript"] = full_text
        st.session_state["video_id"] = video_id
        st.session_state["questions"] = None
        st.success("✅ Transcript extracted!")

if st.session_state.get("transcript"):
    st.video(f"https://www.youtube.com/watch?v={st.session_state[\\"video_id\\"]}")
    st.markdown("---")

    # Step 2: Generate Quiz
    st.header("Step 2 — Generate Quiz")
    num_q = st.slider("Number of questions", 1, 10, 5)
    if st.button("🧠 Generate Questions", type="primary"):
        with st.spinner("Generating..."):
            result, err = call_api("generate", {"transcript": st.session_state["transcript"], "num_questions": num_q})
        if err: st.error(err)
        elif result: st.session_state["questions"] = result["questions"]; st.success(f"✅ {len(result[\\"questions\\"])} questions generated!")

# Step 3: Take Quiz
if st.session_state.get("questions"):
    st.markdown("---")
    st.header("Step 3 — Take the Quiz")
    preference = st.radio("Prefer:", ["videos", "articles"], horizontal=True)
    with st.form("quiz"):
        answers = [st.radio(f"Q{i+1}: {q[\\"question\\"]}", q["choices"], key=f"q_{i}") for i, q in enumerate(st.session_state["questions"])]
        if st.form_submit_button("✅ Submit & Evaluate"):
            with st.spinner("Evaluating..."):
                result, err = call_api("evaluate", {"transcript": st.session_state["transcript"], "questions": st.session_state["questions"], "student_answers": answers, "preference": preference})
            if err: st.error(err)
            elif result:
                st.session_state["eval_result"] = result
                pct = int(result["score"]/result["total"]*100)
                c1,c2,c3 = st.columns(3)
                c1.metric("Score", f"{result[\\"score\\"]}/{result[\\"total\\"]}")
                c2.metric("Percentage", f"{pct}%")
                c3.metric("Status", "✅ Passed" if result["passed"] else "🔄 Review")
                if result["evaluation"]: st.info(result["evaluation"])
                if result["recommendation"]: st.success(result["recommendation"]) if result["passed"] else st.warning(result["recommendation"])

                # Save progress
                call_api("progress/save", {"student_id": student_id, "video_id": st.session_state["video_id"], "video_title": f"Video {st.session_state[\\"video_id\\"]}", "score_raw": result["score"], "score_total": result["total"], "passed": result["passed"], "evaluation_text": result.get("evaluation",""), "recommendation": result.get("recommendation","")})

                # Step 4: Get recommendation
                st.markdown("---")
                st.header("Step 4 — Next Video Recommendation")
                with st.spinner("Finding best next video..."):
                    rec, rec_err = call_api("recommend", {"topic": st.session_state["transcript"][:200], "student_level": "intermediate", "passed": result["passed"], "weak_areas": [], "preference": preference, "watched_video_ids": st.session_state.get("watched_video_ids",[])})
                if rec_err: st.error(rec_err)
                elif rec and rec.get("recommended_video"):
                    rv = rec["recommended_video"]
                    st.success(f"📺 **{rv[\\"title\\"]}**")
                    st.write(f"🔗 [Watch Video]({rv[\\"url\\"]})")
                    st.write(f"💡 {rv[\\"reason\\"]}")
                    if rec.get("alternatives"):
                        st.subheader("Other options:")
                        for alt in rec["alternatives"]:
                            st.write(f"• [{alt[\\"title\\"]}]({alt[\\"url\\"]})")
'''

# Write to file
with open('youtube_transcript_app_integrated.py', 'w') as f:
    f.write(STREAMLIT_CODE)

print("✅ Updated Streamlit app saved to 'youtube_transcript_app_integrated.py'")
print("Run it with: streamlit run youtube_transcript_app_integrated.py")